# Environment Initialization and Data Flash-Extraction

In [ ]:
import os
import shutil
from google.colab import drive

# Mount Google Drive for persistent state tracking storage
drive.mount('/content/drive')

Mounted at /content/drive


# PyTorch Hardware Check

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.11.0+cu128
CUDA: 12.8
CUDA available: True


# Package Installations

In [ ]:
# # 1. Clean out the environment completely
# !pip uninstall -y mamba-ssm causal-conv1d tilelang quack-kernels
# !pip cache purge

# # 2. Make sure Ninja is ready for rapid multi-threading
# !pip install ninja

# # 3. Direct install Python 3.13 compatible versions with the "skip heavy dependencies" flags
# # (This will build using local PyTorch 2.11.0+cu128 in roughly 12-15 minutes total)
# %env MAMBA_SKIP_TILELANG=1
# %env MAMBA_SKIP_CUTLASS=1
# !pip install "causal-conv1d>=1.2.0" --no-build-isolation
# !pip install mamba-ssm==2.3.0 --no-build-isolation

# # 4. Create permanent Drive storage folder path
# !mkdir -p /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/data/mamba_wheels

# # 5. Locate the freshly compiled Python 3.13 wheels from the cache and move them to Drive
# !find /root/.cache/pip/wheels/ -name "*.whl" -exec cp {} /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/data/mamba_wheels/ \;

# # 6. Verify files are safe inside Drive folder
# !ls /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/data/mamba_wheels


In [ ]:
import time

print("Waiting for Google Drive to sync wheels...")
wheel_path = "/content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/data/mamba_wheels/causal_conv1d-1.7.0-cp313-cp313-linux_x86_64.whl"

# Halt execution until the specific file is confirmed to exist
while not os.path.exists(wheel_path):
    time.sleep(1)

# Install the pre-compiled files instantly from Drive
print("Installing mamba_wheels from drive...")
!pip install /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/data/mamba_wheels/*.whl
print("Mamba_wheels installed.")

!pip install -q monai

Waiting for Google Drive to sync wheels...
Installing mamba_wheels from drive...
Processing ./drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/data/mamba_wheels/causal_conv1d-1.7.0-cp313-cp313-linux_x86_64.whl
Processing ./drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/data/mamba_wheels/mamba_ssm-2.3.0-cp313-cp313-linux_x86_64.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 4.7 MB/s eta 0:00:00
Mamba_wheels installed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.6 MB/s eta 0:00:00


# Dataset Flash-Extraction

In [ ]:
# Define raw archive path and fast local runtime target destination paths
DATASET_ZIP = "/content/drive/MyDrive/ML-Datasets/BraTS2020_TrainingData_ABC.zip"
LOCAL_EXTRACT_DIR = "/content/MICCAI_BraTS2020_TrainingData_ABC"
LOCAL_DATA_DIR = LOCAL_EXTRACT_DIR

# Verify archive existence immediately before runtime allocation
assert os.path.exists(DATASET_ZIP), f"Dataset archive not found: {DATASET_ZIP}"

# Robust extraction guard: triggers if directory does not exist or is completely empty
if not os.path.exists(LOCAL_DATA_DIR) or len(os.listdir(LOCAL_DATA_DIR)) == 0:
    print(f"Extracting preprocessed dataset to fast local runtime storage: {LOCAL_EXTRACT_DIR}...")
    os.makedirs(LOCAL_EXTRACT_DIR, exist_ok=True)
    shutil.unpack_archive(DATASET_ZIP, LOCAL_EXTRACT_DIR, "zip")
    print("Extraction complete. Preprocessed dataset ready for I/O operations.")
else:
    print("Valid local preprocessed dataset cache detected. Skipping extraction.")

Extracting preprocessed dataset to fast local runtime storage: /content/MICCAI_BraTS2020_TrainingData_ABC...
Extraction complete. Preprocessed dataset ready for I/O operations.


# Component Integration and Framework Imports

In [ ]:
import sys
import random
import numpy as np
import torch
import torch.optim as optim
import itertools

# Enforce strict scientific reproducibility thresholds across packages
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Optimize CUDA runtime convolution algorithm selection for static tensor patches
torch.backends.cudnn.benchmark = True

# Add src to path
PROJECT_ROOT = "/content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor"
sys.path.append(PROJECT_ROOT)

import src.config as config
from src.dataset import get_brats_dataloaders
from src.losses import SegmentationLoss
from src.expert_engine import run_training

# Instantiate the single-modality sub-network modules
from src.models.mamba_backbone import SingleModalityConvStem3D, OverlappingPatchEmbed3D, BiMambaEncoder3D
from src.models.decoder import AuxiliaryDecoder3D

# Dynamic Module Reloading

In [ ]:
import importlib

importlib.reload(config)

import src.expert_engine as expert_engine
importlib.reload(expert_engine)

import src.models.mamba_backbone as mbb
importlib.reload(mbb)

import src.models.decoder as md
importlib.reload(md)

import src.losses as sl
importlib.reload(sl)

import src.transforms as tf
importlib.reload(tf)

<module 'src.transforms' from '/content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/src/transforms.py'>

# Execution & Orchestration

In [ ]:
from torch.amp import GradScaler

# =========================================================================
# --- TARGET MODALITY CONFIGURATION ---
MODALITY_IDX = 3  # (0: t1, 1: t1ce, 2: t2, 3: flair)
# =========================================================================

MODALITY_NAME = config.MODALITIES[MODALITY_IDX]
SAVE_PATH = config.PRETRAINED_EXPERT_PATHS[MODALITY_NAME]

print(f"===========================================================")
print(f"[*] INITIALIZING EXPERT PRE-TRAINING FOR MODALITY: {MODALITY_NAME.upper()}")
print(f"[*] Checkpoint Target: {SAVE_PATH}")
print(f"===========================================================\n")

# Hardware runtime verification
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Operational Hardware target identified: {device}")

# 1. Pipeline Dataset Loaders Construction
print("Instantiating MONAI dictionary data pipelines...")
train_loader, val_loader = get_brats_dataloaders(batch_size = 16) # defauld set to batch_size=config.BATCH_SIZE

# 2. Structural Module Instantiations
print("Initializing unimodal expert neural net components...")
conv_stem = SingleModalityConvStem3D(out_channels=config.EMBED_DIM).to(device)
patch_embed = OverlappingPatchEmbed3D(patch_size=4, stride=2, in_channels=config.EMBED_DIM, embed_dim=config.EMBED_DIM).to(device)
mamba_backbone = BiMambaEncoder3D(d_model=config.EMBED_DIM, depth=2).to(device)
aux_decoder = AuxiliaryDecoder3D(embed_dim=config.EMBED_DIM, out_channels=config.NUM_SEG_CLASSES).to(device)

model_components = (conv_stem, patch_embed, mamba_backbone, aux_decoder)

# Compute total parameter profile summary metrics for publication tracking
total_params = sum(p.numel() for model in model_components for p in model.parameters())
print(f"Total Single-Modality Expert Trainable Parameters: {total_params:,}")

# 3. Optimization System Setup
criterion = SegmentationLoss().to(device)

# Chain layer parameters
all_parameters = itertools.chain(
    conv_stem.parameters(),
    patch_embed.parameters(),
    mamba_backbone.parameters(),
    aux_decoder.parameters()
)

optimizer = optim.AdamW(
    all_parameters,
    lr=config.PRETRAIN_LR,
    weight_decay=config.WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.PolynomialLR(
    optimizer,
    total_iters=config.PRETRAIN_EPOCHS,
    power=0.9
)

# Device-aware mixed-precision gradient scaling framework
scaler = GradScaler(enabled=(device.type == "cuda"))

# 4. Trigger Orchestration Engine Pipeline
print("Handing execution loop management over to expert stateful runtime engine...")
expert_engine.run_training(
    model_components=model_components,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    scaler=scaler,
    device=device,
    target_modality_idx=MODALITY_IDX,
    save_path=SAVE_PATH,
    session_epoch=90
)

[*] INITIALIZING EXPERT PRE-TRAINING FOR MODALITY: T1
[*] Checkpoint Target: /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/checkpoints/experts/best_expert_t1.pth

Operational Hardware target identified: cuda
Instantiating MONAI dictionary data pipelines...
Initializing unimodal expert neural net components...
Total Single-Modality Expert Trainable Parameters: 1,498,764
Handing execution loop management over to expert stateful runtime engine...
[*] Found existing checkpoint record at: /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/checkpoints/experts/latest_expert_t1.pth. Loading state...
[+] Recovery complete. Resuming from absolute internal epoch counter: 210
[*] Incremental Run Configuration: Training from Epoch 210 -> Target Epoch 300 (+90 epochs)

--- Epoch 211/300 ---


[Train] Seg Loss: 1.5274


[Val] Segmentation Loss-> Mean Dice: 0.4930 (WT: 0.5201, TC: 0.5934, ET: 0.3653)
Stateful tracking saved to: /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/checkpoints/experts/latest_expert_t1.pth

--- Epoch 212/300 ---


[Train] Seg Loss: 1.5106


[Val] Segmentation Loss-> Mean Dice: 0.5874 (WT: 0.6896, TC: 0.6571, ET: 0.4156)
Stateful tracking saved to: /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/checkpoints/experts/latest_expert_t1.pth
*** best expert framework model configuration stored at: /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/checkpoints/experts/best_expert_t1.pth

--- Epoch 213/300 ---


...

[Val] Segmentation Loss-> Mean Dice: 0.5716 (WT: 0.6410, TC: 0.6613, ET: 0.4124)
Stateful tracking saved to: /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/checkpoints/experts/latest_expert_t1.pth

--- Epoch 299/300 ---


[Train] Seg Loss: 1.3894


[Val] Segmentation Loss-> Mean Dice: 0.5713 (WT: 0.6406, TC: 0.6610, ET: 0.4124)
Stateful tracking saved to: /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/checkpoints/experts/latest_expert_t1.pth

--- Epoch 300/300 ---


[Train] Seg Loss: 1.4462


[Val] Segmentation Loss-> Mean Dice: 0.5708 (WT: 0.6396, TC: 0.6612, ET: 0.4115)
Stateful tracking saved to: /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/checkpoints/experts/latest_expert_t1.pth

 Incremental cycle finished successfully. Total absolute epochs processed: 300
